In [37]:
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

In [32]:
# Load cleaned datasets
train = pd.read_csv(r"C:\Users\sanke\Desktop\SwahiliTranslate AI\data\processed\train_cleaned.tsv", sep="\t")
dev = pd.read_csv(r"C:\Users\sanke\Desktop\SwahiliTranslate AI\data\processed\dev_clean.tsv", sep="\t")
test = pd.read_csv(r"C:\Users\sanke\Desktop\SwahiliTranslate AI\data\processed\test_clean.tsv", sep="\t")

# Check dataset sizes
print("Train:", train.shape)
print("Validation:", dev.shape)
print("Test:", test.shape)

Train: (30337, 2)
Validation: (1784, 2)
Test: (1826, 2)


In [33]:
# Loading the pre-trained tokenizer
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set source and target languages
tokenizer.src_lang = "swh_Latn"
tokenizer.tgt_lang = "eng_Latn"

In [34]:
# Select one English-Kiswahili pair
english_sentence = train["en"][0]
swahili_sentence = train["swa"][0]

# Tokenize the Kiswahili sentence
inputs = tokenizer(
    swahili_sentence,
    return_tensors="pt"
)

# Tokenize the English translation
targets = tokenizer(
    text_target=english_sentence,
    return_tensors="pt"
)

print("Kiswahili:", swahili_sentence)
print("English:", english_sentence)

print("\nInput IDs:", inputs["input_ids"])
print("Target IDs:", targets["input_ids"])

Kiswahili: Kinyago cha mtumwa mweusi kilichochongwa na Luiz Paulino da Cunha.
English: Polychromed woodcarving of an Orixá by Luiz Paulino da Cunha.

Input IDs: tensor([[256168,     98,   3442,    562,   1774,  90458,     80,  15675,   1104,
         178048,     65,  30343,     62,   6736,    496,   7156,    791,    170,
          49993,    102, 248075,      2]])
Target IDs: tensor([[256047,   9052,  37682,  13831,     76, 155694,   4158,  14084,    452,
            111,  30978, 248153, 248140,    811,   6736,    496,   7156,    791,
            170,  49993,    102, 248075,      2]])


In [35]:
def tokenize_data(data):
    inputs = tokenizer(
        data["swa"].tolist(),
        max_length=128,
        truncation=True
    )

    targets = tokenizer(
        text_target=data["en"].tolist(),
        max_length=128,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs

In [36]:
train_tokens = tokenize_data(train)
dev_tokens = tokenize_data(dev)
test_tokens = tokenize_data(test)

print("Training examples:", len(train_tokens["input_ids"]))
print("Validation examples:", len(dev_tokens["input_ids"]))
print("Testing examples:", len(test_tokens["input_ids"]))

Training examples: 30337
Validation examples: 1784
Testing examples: 1826


In [38]:
#converting the tokenized data into Hugging Face datasets
train_dataset = Dataset.from_dict(train_tokens)
dev_dataset = Dataset.from_dict(dev_tokens)
test_dataset = Dataset.from_dict(test_tokens)

# Create a DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": dev_dataset,
    "test": test_dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 30337
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1784
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1826
    })
})


In [39]:
#preparing the model for fine tuning
from transformers import AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 10528.79it/s]


In [40]:
print(type(model))

<class 'transformers.models.m2m_100.modeling_m2m_100.M2M100ForConditionalGeneration'>
